# RELEVO — Pipeline de Computação Gráfica Passo a Passo (F1-T10)

Este notebook demonstra cada etapa algorítmica do **RELEVO** (100 % Computação Gráfica clássica, sem IA):
1. **Segmentação por cor de fundo + morfologia matemática** (`F1-T06`)
2. **Rastreamento de contorno (Suzuki–Abe e Moore) + simplificação de Douglas–Peucker** (`F1-T12`)
3. **Alinhamento afim 2D e matriz 3×3 grade → foto** (`F1-T04`)
4. **Visual Hull + hipótese de seção transversal + campo de distância assinada (SDF)** (`F0-T03`, `ADR 0005`)
5. **Extração de isosuperfície por Marching Cubes + Suavização de Taubin ($\lambda|\mu$)** (`F1-T02`)
6. **Simplificação por erro quádrico de Garland–Heckbert** (`F1-T03`)
7. **Colorização por vértice ponderada por normal (Lambert) e Mapeamento UV em 6 ilhas** (`F1-T05`, `F4-T06`)

In [ ]:
import matplotlib.pyplot as plt
from relevo_pipeline import processar
from relevo_pipeline.contorno import rastrear_contorno, rastrear_moore, simplificar
from relevo_pipeline.malha import gerar_lods
from relevo_pipeline.segmentacao import gerar_mascara
from relevo_pipeline.sintetico import gerar_fotos
from relevo_pipeline.textura import assar_textura

fotos = gerar_fotos()
print('Vistas geradas:', list(fotos.keys()), [v.shape for v in fotos.values()])

In [ ]:
# 1. Segmentação + Rastreamento de Contorno (Suzuki-Abe vs Moore + Douglas-Peucker)
mascara_lat = gerar_mascara(fotos['lateral'])
contorno_denso = rastrear_contorno(mascara_lat)
contorno_moore = rastrear_moore(mascara_lat)
contorno_simpl = simplificar(contorno_denso, tolerancia_px=1.5)
print(
    f'Densos: {len(contorno_denso)} | Moore: {len(contorno_moore)} '
    f'| Simplificado: {len(contorno_simpl)}'
)

fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
axs[0].imshow(fotos['lateral'][..., ::-1])
axs[0].set_title('Foto Lateral (RGB)')
axs[1].imshow(mascara_lat, cmap='gray')
axs[1].set_title('Máscara binária')
axs[2].imshow(fotos['lateral'][..., ::-1])
axs[2].plot(contorno_simpl[:, 0], contorno_simpl[:, 1], '-o', color='#ff6b3d', markersize=3)
axs[2].set_title(f'Contorno ({len(contorno_simpl)} vértices)')
for ax in axs:
    ax.axis('off')
plt.tight_layout()

In [ ]:
# 2. Pipeline completo: Alinhamento -> Visual Hull -> Marching Cubes -> Taubin -> QEM -> Cor
resultado = processar(
    fotos['lateral'],
    fotos['topo'],
    fotos['frente'],
    comprimento_cm=28.0,
    resolucao=96,
    faces_alvo=10_000,
)
m = resultado.metricas
print('Dimensões (C × A × L cm):', m['dimensoes_cm'])
print('Suavização Taubin:', m['suavizacao'])
print('Decimação Garland-Heckbert:', m['decimacao'])
print('Consistência (IoU voxels + IoU malha):', m['consistencia'])

In [ ]:
# 3. Mapeamento UV em 6 ilhas (F4-T06) e Níveis de Detalhe LOD (F5-T02)
malha_uv, textura_rgb, info_uv = assar_textura(
    resultado.malha, fotos, resultado.vistas, tamanho=512
)
lods = gerar_lods(resultado.malha)
print('Atlas UV:', info_uv)
print('Níveis LOD:', [(n['nivel'], n['faces'], f"{n['erro_medio_mm']} mm") for n in lods])

plt.figure(figsize=(5, 5))
plt.imshow(textura_rgb)
plt.title('Atlas de Textura UV 512×512 (6 ilhas + dilatação)')
plt.axis('off')
plt.show()